# 01 hnscc_analysis_framework_LOCKED_with_NIMRAD

Source code preserved from the supplied analysis notebook. Saved outputs and execution counts have been cleared for public distribution. See README for input requirements and limitations.


In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test
import matplotlib.pyplot as plt


In [ ]:
HYPOXIA_COEFFICIENTS = {
    "CP":   0.27180250,
    "GPC3": 0.09274532,
    "STC1": 0.01305204,
}

RS_COEFFICIENTS = {
    "ATR":    0.004630324,
    "RPA2":   0.334318088,
    "BLM":    0.224064815,
    "MRE11A": 0.004784271,
}

USING_PLACEHOLDER_COEFFICIENTS = False

if USING_PLACEHOLDER_COEFFICIENTS:
    print("Placeholder equal-weight coefficients in use -- do not interpret results below.")
else:
    print("Locked real coefficients in use (received from Jimmy).")

In [ ]:
TOUSTRUP_GENES = [
    "ADM", "ALDOA", "ANKRD37", "BNIP3", "BNIP3L", "FAM162A", "EGLN3",
    "KCTD11", "LOX", "NDRG1", "P4HA1", "P4HA2", "PDK1", "PFKFB3", "SLC2A1"
]

TOUSTRUP_COEFFICIENTS = {gene: 1/15 for gene in TOUSTRUP_GENES}

# NIMRAD / Eustace 26-gene HNSCC hypoxia signature -- RESOLVED against the
# actual primary paper (Eustace et al., Clin Cancer Res 2013, Table 1 and
# Methods), not a secondary summary. Both earlier open questions are now
# settled:
#
# 1. Gene list: Table 1 confirms all 26 genes explicitly. HIG2 is correct
#    (a secondary AI-generated summary had incorrectly substituted PPM1J --
#    PPM1J is not part of the real panel and has been removed). The paper's
#    own Table 1 has a likely typo, "MPRS17" -- this is MRPS17 (mitochondrial
#    ribosomal protein S17), a real, well-established gene, now included.
#    Aliases resolved to current HGNC symbols: HIG2 -> HILPDA, C20orf20 -> MRGBP.
#    ANGPTL4 and KRT17 are confirmed absent from this HNSC expression dataset
#    and are excluded (24 of 26 genes usable here).
#
# 2. Reference/control genes: the Methods state plainly that only 3 genes were
#    actually used for normalisation -- "the geometric mean of 3 reference
#    genes (GNB2L1, B2M, and RPL11)" -- selected as the best-performing 3 of
#    5 originally tested candidates. The 2 extra candidates (HPRT1, RPL24)
#    were NOT part of the final calculation and have been removed below.
NIMRAD_GENES = [
    "ALDOA", "ANLN", "BNC1", "CA9", "CDKN3", "COL4A6", "DCBLD1", "ENO1",
    "FAM83B", "FOSL1", "GNAI1", "HILPDA", "KCTD11", "LDHA", "P4HA1",
    "PGAM1", "PGK1", "SDC1", "SLC16A1", "SLC2A1", "TPI1", "VEGFA", "MRGBP",
    "MRPS17"
]

NIMRAD_COEFFICIENTS = {gene: 1/len(NIMRAD_GENES) for gene in NIMRAD_GENES}

# Corrected: only the 3 genes actually used in the original paper's final
# normalisation (verified directly from the Methods section).
NIMRAD_CONTROL_GENES = ["GNB2L1", "B2M", "RPL11"]

In [ ]:
columns_we_need = [
    "Patient ID", "Sample ID", "Diagnosis Age",
    "Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code",
    "American Joint Committee on Cancer Tumor Stage Code",
    "Hpv status p16",
    "Overall Survival (Months)", "Overall Survival Status",
    "Did patient start adjuvant postoperative radiotherapy?",
    "Patient Smoking History Category",
    "CP", "GPC3", "STC1", "ATR", "RPA2", "BLM", "MRE11A"
] + TOUSTRUP_GENES + NIMRAD_GENES + NIMRAD_CONTROL_GENES

df = pd.read_csv("clinical_rna_fire_combine.csv", usecols=columns_we_need)

df["is_primary_tumour"] = df["Sample ID"].str.endswith("-01")
df = df[df["is_primary_tumour"] == True]

gene_list = ["CP", "GPC3", "STC1", "ATR", "RPA2", "BLM", "MRE11A"] + TOUSTRUP_GENES + NIMRAD_GENES + NIMRAD_CONTROL_GENES
gene_list = list(dict.fromkeys(gene_list))  # de-duplicate (a few genes overlap Toustrup/NIMRAD)
for gene in gene_list:
    df[gene] = df[gene].astype(float)
    df[gene] = np.log2(df[gene] + 1)

print("Cohort after primary-tumour filter and log2 transform:", df.shape[0], "patients")

In [ ]:
def calculate_score(data, coefficient_dict):
    weighted_sum = 0
    for gene, weight in coefficient_dict.items():
        z = (data[gene] - data[gene].mean()) / data[gene].std()
        weighted_sum = weighted_sum + (z * weight)
    return weighted_sum

def calculate_median_control_score(data, panel_genes, control_genes):
    # Closer structural match to the original NIMRAD/Eustace algorithm:
    # median(panel) minus mean(controls), computed in z-score space instead of
    # qPCR Ct space (Ct and log2-expression run in opposite directions, so the
    # sign here is oriented so higher score = more hypoxic, matching the
    # original score's direction).
    panel_z = pd.DataFrame({g: (data[g] - data[g].mean()) / data[g].std() for g in panel_genes})
    control_z = pd.DataFrame({g: (data[g] - data[g].mean()) / data[g].std() for g in control_genes})
    return panel_z.median(axis=1) - control_z.mean(axis=1)

df["HypoxiaScore"] = calculate_score(df, HYPOXIA_COEFFICIENTS)
df["RSScore"] = calculate_score(df, RS_COEFFICIENTS)
df["ToustrupScore"] = calculate_score(df, TOUSTRUP_COEFFICIENTS)
df["NimradScore"] = calculate_score(df, NIMRAD_COEFFICIENTS)
df["NimradScore_v2"] = calculate_median_control_score(df, NIMRAD_GENES, NIMRAD_CONTROL_GENES)

df["HypoxiaScore_c"] = df["HypoxiaScore"] - df["HypoxiaScore"].mean()
df["RSScore_c"] = df["RSScore"] - df["RSScore"].mean()
df["ToustrupScore_c"] = df["ToustrupScore"] - df["ToustrupScore"].mean()
df["NimradScore_c"] = df["NimradScore"] - df["NimradScore"].mean()
df["NimradScore_v2_c"] = df["NimradScore_v2"] - df["NimradScore_v2"].mean()

df[["HypoxiaScore_c", "RSScore_c", "ToustrupScore_c", "NimradScore_c", "NimradScore_v2_c"]].describe()

In [ ]:
df["OS_months"] = df["Overall Survival (Months)"]
df["Event"] = df["Overall Survival Status"].str.startswith("1").astype(int)
df = df.dropna(subset=["OS_months", "Event"])
df = df[df["OS_months"] > 0]

print("Final analysis cohort:", df.shape[0], "patients,", df["Event"].sum(), "events")

In [ ]:
df["Age"] = df["Diagnosis Age"]

t_stage_map = {
    "T1": "T1-2", "T2": "T1-2",
    "T3": "T3-4", "T4": "T3-4", "T4a": "T3-4", "T4b": "T3-4"
}
df["T_group"] = df["American Joint Committee on Cancer Tumor Stage Code"].map(t_stage_map)
df["T_group"] = df["T_group"].fillna("Unknown")

n_stage_map = {
    "N0": "N0",
    "N1": "N+", "N2": "N+", "N2a": "N+", "N2b": "N+", "N2c": "N+", "N3": "N+"
}
df["N_group"] = df["Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code"].map(n_stage_map)
df["N_group"] = df["N_group"].fillna("Unknown")

df["Smoking_group"] = "Ever"
df.loc[df["Patient Smoking History Category"] == "Lifelong Non-Smoker", "Smoking_group"] = "Never"
df.loc[df["Patient Smoking History Category"].isna(), "Smoking_group"] = "Unknown"

df["HPV_p16"] = df["Hpv status p16"]

df["RT_status"] = df["Did patient start adjuvant postoperative radiotherapy?"]
df["RT_status"] = df["RT_status"].replace({"YES": "Yes", "NO": "No"})

In [ ]:
print("Total patients:", df.shape[0])
print("Total events (deaths):", int(df["Event"].sum()))
print("Median follow-up (months):", round(df["OS_months"].median(), 1))

In [ ]:
missing_rows = []

def add_missing_row(column_name, label):
    n_total = df.shape[0]
    n_missing = df[column_name].isna().sum()
    n_present = n_total - n_missing
    missing_rows.append({
        "Variable": label,
        "N recorded": n_present,
        "% recorded": round(100 * n_present / n_total, 1),
        "N missing": n_missing,
        "% missing": round(100 * n_missing / n_total, 1),
    })

add_missing_row("Age", "Age")
add_missing_row("HPV_p16", "HPV/p16 status")
add_missing_row("American Joint Committee on Cancer Tumor Stage Code", "T stage (raw)")
add_missing_row("Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code", "N stage (raw)")
add_missing_row("Patient Smoking History Category", "Smoking history")
add_missing_row("RT_status", "RT status")

missing_table = pd.DataFrame(missing_rows)
missing_table

In [ ]:
missing_table.to_csv("hnsc_cohort_summary_missingness.csv", index=False)

In [ ]:
age_summary = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Min", "Max"],
    "Age (years)": [
        round(df["Age"].mean(), 1),
        df["Age"].median(),
        df["Age"].min(),
        df["Age"].max(),
    ]
})
age_summary

In [ ]:
category_rows = []

def add_category_rows(column_name, variable_label):
    n_total = df.shape[0]
    counts = df[column_name].value_counts(dropna=False)
    for category_value, n in counts.items():
        category_rows.append({
            "Variable": variable_label,
            "Category": category_value,
            "N": n,
            "%": round(100 * n / n_total, 1),
        })

add_category_rows("HPV_p16", "HPV/p16 status")
add_category_rows("T_group", "T stage group")
add_category_rows("N_group", "N stage group")
add_category_rows("Smoking_group", "Smoking group")
add_category_rows("RT_status", "RT status")

category_table = pd.DataFrame(category_rows)
category_table

In [ ]:
rt_defined_subset = df[df["RT_status"].isin(["Yes", "No"])].copy()

rt_subset_summary = pd.DataFrame({
    "Category": ["Total in subset", "RT = Yes", "RT = No", "Events (deaths)"],
    "N": [
        rt_defined_subset.shape[0],
        (rt_defined_subset["RT_status"] == "Yes").sum(),
        (rt_defined_subset["RT_status"] == "No").sum(),
        int(rt_defined_subset["Event"].sum()),
    ]
})

rt_defined_subset.to_csv("hnsc_RT_defined_subset.csv", index=False)
rt_subset_summary

In [ ]:
def run_univariable_cox(data, score_col, label):
    print(label)
    d = data[["OS_months", "Event", score_col]].dropna()
    cph = CoxPHFitter()
    cph.fit(d, duration_col="OS_months", event_col="Event")
    cph.print_summary()
    print("C-index:", cph.concordance_index_)
    return cph

def plot_km_median_split(data, score_col, label):
    median = data[score_col].median()
    high = data[data[score_col] >= median]
    low = data[data[score_col] < median]

    kmf_high = KaplanMeierFitter().fit(high["OS_months"], high["Event"], label=f"High {label}")
    kmf_low = KaplanMeierFitter().fit(low["OS_months"], low["Event"], label=f"Low {label}")

    plt.figure(figsize=(7, 5.5))
    kmf_high.plot_survival_function()
    kmf_low.plot_survival_function()
    plt.xlabel("Overall Survival (Months)")
    plt.ylabel("Survival Probability")
    plt.title(f"OS by {label} Group")
    plt.grid(alpha=0.3)
    plt.show()

    lr = logrank_test(high["OS_months"], low["OS_months"], high["Event"], low["Event"])
    print("Log-rank p-value:", lr.p_value)

In [ ]:
cph_hypoxia = run_univariable_cox(df, "HypoxiaScore_c", "Hypoxia score (locked coefficients)")

In [ ]:
plot_km_median_split(df, "HypoxiaScore_c", "Hypoxia score")

In [ ]:
cph_rs = run_univariable_cox(df, "RSScore_c", "RS score (locked coefficients, full cohort -- secondary analysis)")

In [ ]:
plot_km_median_split(df, "RSScore_c", "RS score")

In [ ]:
cph_toustrup = run_univariable_cox(df, "ToustrupScore_c", "Toustrup score (secondary benchmark)")

In [ ]:
plot_km_median_split(df, "ToustrupScore_c", "Toustrup score")

In [ ]:
cph_nimrad = run_univariable_cox(df, "NimradScore_c", "NIMRAD v1 -- simple equal-weighted (secondary benchmark)")

In [ ]:
plot_km_median_split(df, "NimradScore_c", "NIMRAD v1 (simple)")

In [ ]:
cph_nimrad_v2 = run_univariable_cox(df, "NimradScore_v2_c", "NIMRAD v2 -- control-normalised (secondary benchmark)")

In [ ]:
plot_km_median_split(df, "NimradScore_v2_c", "NIMRAD v2 (control-normalised)")

In [ ]:
SCORES = [
    ("HypoxiaScore_c", "Hypoxia score"),
    ("RSScore_c", "RS score"),
    ("ToustrupScore_c", "Toustrup score"),
    ("NimradScore_c", "NIMRAD v1 score"),
    ("NimradScore_v2_c", "NIMRAD v2 score"),
]

def build_design(data, score_col, extra_covariate=None, extra_ref=None):
    covariates = ["Age", "T_group", "N_group", "Smoking_group"]
    dummy_cols = ["T_group", "N_group", "Smoking_group"]
    ref_cols = ["T_group_T1-2", "N_group_N0", "Smoking_group_Never"]

    if extra_covariate:
        covariates.append(extra_covariate)
        dummy_cols.append(extra_covariate)
        ref_cols.append(extra_ref)

    d = data[["OS_months", "Event", score_col] + covariates].copy()
    d = pd.get_dummies(d, columns=dummy_cols)
    d = d.drop(columns=[c for c in ref_cols if c in d.columns])

    unknown_cols = [c for c in d.columns if c.endswith("_Unknown")]
    if unknown_cols:
        d = d[d[unknown_cols].sum(axis=1) == 0].drop(columns=unknown_cols)

    return d.dropna()

def fit_adjusted_model(data, score_col, label, extra_covariate=None, extra_ref=None):
    design = build_design(data, score_col, extra_covariate, extra_ref)
    print(f"{label} — N = {design.shape[0]}, events = {int(design['Event'].sum())}")
    cph = CoxPHFitter(penalizer=0.001)
    cph.fit(design, duration_col="OS_months", event_col="Event")
    cph.print_summary()
    print("C-index:", cph.concordance_index_)
    print()
    return cph

In [ ]:
for score_col, label in SCORES:
    fit_adjusted_model(df, score_col, f"Primary model: {label}")

In [ ]:
hpv_known = df[df["HPV_p16"].isin(["Positive", "Negative"])].copy()
print("HPV/p16-known subset: N =", hpv_known.shape[0], "| events =", int(hpv_known["Event"].sum()))
print()

for score_col, label in SCORES:
    fit_adjusted_model(hpv_known, score_col, f"HPV-known sensitivity model: {label}",
                        extra_covariate="HPV_p16", extra_ref="HPV_p16_Negative")

In [ ]:
rt_known = df[df["RT_status"].isin(["Yes", "No"])].copy()
print("RT-known subset: N =", rt_known.shape[0], "| events =", int(rt_known["Event"].sum()))
print()

for score_col, label in SCORES:
    fit_adjusted_model(rt_known, score_col, f"RT-known sensitivity model: {label}",
                        extra_covariate="RT_status", extra_ref="RT_status_No")

In [ ]:
rt_yes_only = df[df["RT_status"] == "Yes"].copy()
rt_yes_only["RSScore_main"] = calculate_score(rt_yes_only, RS_COEFFICIENTS)
rt_yes_only["RSScore_main_c"] = rt_yes_only["RSScore_main"] - rt_yes_only["RSScore_main"].mean()

run_univariable_cox(rt_yes_only, "RSScore_main_c",
                     "RS score -- RT-treated patients only, MAIN analysis, locked coefficients (N=119)")

In [ ]:
plot_km_median_split(rt_yes_only, "RSScore_main_c", "RS score (RT-treated only)")

In [ ]:
rt_defined_for_interaction = df[df["RT_status"].isin(["Yes", "No"])].copy()

# Single shared z-score across both RT groups together (not computed separately)
rt_defined_for_interaction["RSScore_shared"] = calculate_score(rt_defined_for_interaction, RS_COEFFICIENTS)
rt_defined_for_interaction["RSScore_shared_c"] = (
    rt_defined_for_interaction["RSScore_shared"] - rt_defined_for_interaction["RSScore_shared"].mean()
)

rt_defined_for_interaction["RT_numeric"] = (rt_defined_for_interaction["RT_status"] == "Yes").astype(int)
rt_defined_for_interaction["RS_x_RT"] = (
    rt_defined_for_interaction["RSScore_shared_c"] * rt_defined_for_interaction["RT_numeric"]
)

interaction_design = rt_defined_for_interaction[
    ["OS_months", "Event", "RSScore_shared_c", "RT_numeric", "RS_x_RT"]
].dropna()

print("RS x RT interaction model -- N =", interaction_design.shape[0],
      "| events =", int(interaction_design["Event"].sum()))

cph_interaction = CoxPHFitter()
cph_interaction.fit(interaction_design, duration_col="OS_months", event_col="Event")
cph_interaction.print_summary()
print("C-index:", cph_interaction.concordance_index_)

In [ ]:
print("See Section 8 for the full-514-patient RS result (now: secondary analysis).")
print("RS score (full cohort) HR, CI, p, and C-index are reported there and are unchanged.")